Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: CCS4310 – Deep Learning  
Stage: EDA  
Purpose: Audit real DeepFashion images, captions, and identifier matching.

# DeepFashion-MultiModal Fashion Generation EDA

A read-only, reproducible audit of the local dataset.


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: CCS4310 – Deep Learning  
Stage: EDA  
Purpose: Audit images, captions, and matching. With FAST_DEV_RUN=True, image statistics and visual examples are sample-based rather than full-dataset estimates.

## Configuration and paths

In [ ]:
from pathlib import Path
import pathlib
import json, random
import numpy as np, pandas as pd
PROJECT_ROOT=Path.cwd()
for candidate in [PROJECT_ROOT,*PROJECT_ROOT.parents]:
    if (candidate/"data"/"raw"/"deepfashion").exists(): PROJECT_ROOT=candidate; break
DEEPFASHION_DIR=PROJECT_ROOT/"data"/"raw"/"deepfashion"; IMAGES_DIR=DEEPFASHION_DIR/"images"
CAPTIONS_PATH=DEEPFASHION_DIR/"captions.json"; OUTPUT_FIGURES_DIR=PROJECT_ROOT/"outputs"/"figures"
IMAGE_EXTENSIONS={".jpg",".jpeg",".png",".webp"}; SEED=42; random_state=SEED; FAST_DEV_RUN=True; MAX_IMAGES=1000; FAST_DEV_MAX_IMAGES=MAX_IMAGES
random.seed(SEED); np.random.seed(SEED); OUTPUT_FIGURES_DIR.mkdir(parents=True,exist_ok=True)
print("captions path:",CAPTIONS_PATH,"exists:",CAPTIONS_PATH.exists())


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: EDA  
Purpose: Audit images, captions, and matching.

## Reusable caption normalization and structure-derived matching

In [ ]:
def _caption_text(value):
    if isinstance(value, str): return value.strip()
    if isinstance(value, (list, tuple)): return " ".join(filter(None, (_caption_text(x) for x in value))).strip()
    if isinstance(value, dict):
        for key in ("text", "caption", "description", "sentence", "prompt"):
            if key in value:
                text = _caption_text(value[key])
                if text: return text
    return ""

_ID_KEYS=("image", "image_path", "image_name", "filename", "file_name", "file", "path", "id", "image_id", "name")
_TEXT_KEYS=("caption", "captions", "text", "description", "sentence", "prompt")
def _normal_key(value):
    return str(value).replace("\\", "/").lstrip("./")
def _walk_caption_objects(obj, context=""):
    """Yield only evidence-backed (identifier, caption) pairs from common JSON layouts."""
    if isinstance(obj, dict):
        identifier = next((obj[k] for k in _ID_KEYS if k in obj and isinstance(obj[k], (str, int))), None)
        text = next((_caption_text(obj[k]) for k in _TEXT_KEYS if k in obj), "")
        if identifier is not None and text: yield _normal_key(identifier), text
        # Mapping layout: {filename: caption/list/object}.
        for key, value in obj.items():
            key_text = str(key)
            value_text = _caption_text(value)
            looks_like_id = pathlib.PurePosixPath(key_text.replace("\\", "/")).suffix.lower() in IMAGE_EXTENSIONS or "/" in key_text or "\\" in key_text
            if (value_text or looks_like_id) and (looks_like_id or context == ""):
                yield _normal_key(key_text), value_text
            elif isinstance(value, (dict, list)):
                yield from _walk_caption_objects(value, key_text)
    elif isinstance(obj, list):
        for value in obj: yield from _walk_caption_objects(value, context)

def normalize_caption_records(raw):
    if not isinstance(raw, (dict, list)):
        raise ValueError("Unsupported caption structure: expected a mapping or list of records")
    records=[]
    for identifier, caption in _walk_caption_objects(raw):
        records.append({"image_identifier": _normal_key(identifier), "caption": caption})
    if not records: raise ValueError("Unsupported caption structure: no identifier/text records found")
    return pd.DataFrame(records).drop_duplicates(["image_identifier", "caption"]).reset_index(drop=True)

def caption_for_path(path, records):
    path=path if isinstance(path, pathlib.PurePosixPath) else pathlib.PurePosixPath(str(path).replace("\\", "/"))
    if isinstance(records, pd.DataFrame):
        pairs=records[["image_identifier","caption"]].itertuples(index=False)
        records={str(k):str(v) for k,v in pairs}
    candidates={path.as_posix(), path.name, path.stem}
    exact=[v for k,v in records.items() if k in candidates]
    if exact: return exact[0]
    basename=[v for k,v in records.items() if pathlib.PurePosixPath(k).name == path.name]
    if len(basename)==1: return basename[0]
    stem=[v for k,v in records.items() if pathlib.PurePosixPath(k).stem == path.stem]
    return stem[0] if len(stem)==1 else ""


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: EDA  
Purpose: Audit images, captions, and matching.

## Dataset status and caption matching

In [ ]:
image_paths=sorted(p for p in IMAGES_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
raw=json.loads(CAPTIONS_PATH.read_text(encoding="utf-8")) if CAPTIONS_PATH.exists() else {}
caption_records=normalize_caption_records(raw)
caption_exact=dict(zip(caption_records.image_identifier,caption_records.caption))
caption_by_name={}; caption_by_stem={}
for identifier,caption in caption_exact.items():
    key=pathlib.PurePosixPath(identifier); caption_by_name.setdefault(key.name,[]).append(caption); caption_by_stem.setdefault(key.stem,[]).append(caption)
def fast_caption_for_path(path):
    key=pathlib.PurePosixPath(str(path).replace('\\', '/'))
    if key.as_posix() in caption_exact: return caption_exact[key.as_posix()]
    if key.name in caption_by_name and len(caption_by_name[key.name])==1: return caption_by_name[key.name][0]
    if key.stem in caption_by_stem and len(caption_by_stem[key.stem])==1: return caption_by_stem[key.stem][0]
    return ''
matched_all={p:fast_caption_for_path(p.relative_to(IMAGES_DIR)) for p in image_paths}
caption_image_matches={identifier for identifier in caption_exact if pathlib.PurePosixPath(identifier).name in {p.name for p in image_paths}}
matched_caption_records=sum(1 for identifier in caption_records.image_identifier if pathlib.PurePosixPath(identifier).name in {p.name for p in image_paths})
unmatched_caption_records=len(caption_records)-matched_caption_records
inspect_paths=(random.Random(SEED).sample(image_paths,min(FAST_DEV_MAX_IMAGES,len(image_paths))) if FAST_DEV_RUN else image_paths)
matched_sample={p:matched_all[p] for p in inspect_paths}
matched_visual_paths=[p for p in inspect_paths if matched_all[p]]
matched_count=sum(bool(x) for x in matched_all.values())
status=pd.DataFrame({"item":["total images","caption records","matched caption records","unmatched caption records","images without captions"],"count":[len(image_paths),len(caption_records),matched_caption_records,unmatched_caption_records,len(image_paths)-matched_count]})
display(status)


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: EDA  
Purpose: Audit images, captions, and matching.

## Random sample, caption statistics, and one-pass image properties

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
rng=random.Random(random_state); sample_paths=rng.sample(matched_visual_paths,min(16,len(matched_visual_paths)))
rows=[]; corrupt=[]; sample_cache={}
for path in inspect_paths:  # one open per selected image; cache sampled pixels for plots
    try:
        with Image.open(path) as im:
            rgb=im.convert("RGB")
            rows.append({"image_path":str(path.relative_to(IMAGES_DIR)),"width":im.width,"height":im.height,"aspect_ratio":im.width/im.height,"mode":im.mode,"caption":matched_sample[path],"image_identifier":path.relative_to(IMAGES_DIR).as_posix(),"is_valid":True})
            if path in sample_paths: sample_cache[path]=rgb.copy()
    except Exception as exc:
        corrupt.append({"image_path":str(path.relative_to(IMAGES_DIR)),"error":str(exc)})
        rows.append({"image_path":str(path.relative_to(IMAGES_DIR)),"width":np.nan,"height":np.nan,"aspect_ratio":np.nan,"mode":"invalid","caption":matched_sample[path],"image_identifier":path.relative_to(IMAGES_DIR).as_posix(),"is_valid":False})
image_properties=pd.DataFrame(rows)
if not image_properties.empty:
    image_properties["caption_words"]=image_properties.caption.fillna("").str.split().str.len()
    display(image_properties.head()); display(image_properties[["width","height","aspect_ratio","caption_words"]].describe())
print("corrupt:",len(corrupt)); print("caption examples:"); display(image_properties.loc[image_properties.caption.ne(""),["image_path","caption"]].head(5))


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: EDA  
Purpose: Audit images, captions, and matching.

## Required plots and visual sample

In [ ]:
if not image_properties.empty:
    fig,ax=plt.subplots(1,3,figsize=(16,4)); image_properties.width.dropna().hist(ax=ax[0],bins=30); image_properties.height.dropna().hist(ax=ax[1],bins=30); image_properties.aspect_ratio.dropna().hist(ax=ax[2],bins=30); ax[0].set_title("Image width"); ax[1].set_title("Image height"); ax[2].set_title("Aspect ratio (width / height)"); fig.suptitle("DeepFashion image distributions (sample-based in FAST_DEV_RUN)"); fig.tight_layout(); fig.savefig(OUTPUT_FIGURES_DIR/"maleesha_deepfashion_eda_distributions.png",dpi=150); plt.show()
fig,axes=plt.subplots(4,4,figsize=(12,12)); axes=axes.ravel()
for ax,path in zip(axes,sample_paths):
    image=sample_cache.get(path)
    if image is not None: ax.imshow(image)
    ax.set_title(f"{path.name[:18]}\n{matched_sample.get(path, 'No caption')[:45]}"); ax.axis("off")
for ax in axes[len(sample_paths):]: ax.axis("off")
fig.suptitle("Reproducible random image sample (seed 42)"); fig.tight_layout(); fig.savefig(OUTPUT_FIGURES_DIR/"maleesha_deepfashion_eda_random_sample.png",dpi=150); plt.show()


Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: EDA  
Purpose: Audit images, captions, and matching.

## Suitability summary

In [ ]:
summary={"total_images":len(image_paths),"analysed_images":len(inspect_paths),"valid_images":int(image_properties.get("is_valid",pd.Series(dtype=bool)).sum()),"corrupt_images":len(corrupt),"caption_records":len(caption_records),"matched_images":matched_count,"images_without_captions":len(image_paths)-matched_count,"unmatched_captions":max(0,len(caption_records)-matched_count),"fast_dev_run":FAST_DEV_RUN}
display(pd.DataFrame([summary])); print("No raw files were modified.")

Owner: Maleesha  
Project: Explainable-Fashion-Design-AI  
Module: EDA  
Purpose: Audit images, captions, and matching.

## Limitations and conclusion
Caption schemas are inferred from observed keys and may contain ambiguous identifiers; matching is conservative and basename/stem based. Image statistics cover the selected random sample in fast mode and all files otherwise.

The conclusion should be read after execution: use the displayed `summary` and tables to decide whether caption coverage, empty captions, image corruption, and aspect-ratio variation are acceptable for preprocessing. No training-quality or generation-performance claim is made.